In [41]:
# Install PySpark so the notebook can run Spark code in this environment.
# findspark helps Python locate and initialize the Spark installation.
!pip install pyspark findspark


Defaulting to user installation because normal site-packages is not writeable


This cell handles the installation of the necessary libraries to run Spark within a Python environment. pyspark is the Spark Python API, and findspark is used to locate the Spark installation on the system and initialize it.

In [42]:
# Import findspark so Python can locate the local Spark installation.
import findspark

# Initialize Spark paths before importing or creating a SparkSession.
findspark.init()

# Import SparkSession to create the Spark application.
# Import Window for lag/rolling-window calculations later in the notebook.
from pyspark.sql import SparkSession, Window

# Import PySpark SQL functions using alias F for cleaner column expressions.
from pyspark.sql import functions as F

# Import schema/data type classes so the CSV can be read with explicit column types.
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Create a Spark session for the airline delay analysis project.
# The memory settings give Spark more room for processing the dataset.
spark = SparkSession.builder \
    .appName("Airline_Delay_Project_PySpark") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

# Display the SparkSession object to confirm Spark started successfully.
spark


Here, the SparkSession is initialized. This is the entry point to programming Spark with the Dataset and DataFrame API. The configuration allocates 4GB of memory to both the driver and executors to ensure the environment can handle the airline dataset efficiently.

In [43]:
# Define the expected schema for the airline delay CSV file.
# Using an explicit schema helps Spark read columns with the correct data types.
schema = StructType([
    # Year and month identify the time period for each record.
    StructField("year", IntegerType(), True),
    StructField("month", IntegerType(), True),

    # Carrier and airport fields identify the airline and airport being analyzed.
    StructField("carrier", StringType(), True),
    StructField("carrier_name", StringType(), True),
    StructField("airport", StringType(), True),
    StructField("airport_name", StringType(), True),

    # Flight count and delayed-flight count fields.
    StructField("arr_flights", DoubleType(), True),
    StructField("arr_del15", DoubleType(), True),

    # Delay incident counts by cause.
    StructField("carrier_ct", DoubleType(), True),
    StructField("weather_ct", DoubleType(), True),
    StructField("nas_ct", DoubleType(), True),
    StructField("security_ct", DoubleType(), True),
    StructField("late_aircraft_ct", DoubleType(), True),

    # Cancelled, diverted, and total delay-minute fields.
    StructField("arr_cancelled", DoubleType(), True),
    StructField("arr_diverted", DoubleType(), True),
    StructField("arr_delay", DoubleType(), True),

    # Delay minutes broken down by cause.
    StructField("carrier_delay", DoubleType(), True),
    StructField("weather_delay", DoubleType(), True),
    StructField("nas_delay", DoubleType(), True),
    StructField("security_delay", DoubleType(), True),
    StructField("late_aircraft_delay", DoubleType(), True)
])


This cell explicitly defines the data schema using StructType. By pre-defining the data types (Integer, String, Double) for each column like arr_flights and weather_delay, we ensure data integrity and improve the performance of the data loading process.

In [44]:
# Read the cleaned airline delay CSV into a Spark DataFrame.
# header=True tells Spark the first row contains column names.
# schema=schema forces Spark to use the data types defined above.
df = spark.read.csv("airline_delay_cleaned2.csv", header=True, schema=schema)


The dataset is loaded into a Spark DataFrame. By using the previously defined schema and setting header=True, Spark correctly maps the CSV values to their corresponding column names and types.

In [45]:
# Check for missing values in a few important columns.
# count('*') counts all rows, while count(column) ignores nulls.
# The difference gives the number of missing values for that column.
df.select(
    (F.count('*') - F.count('year')).alias('missing_year'),
    (F.count('*') - F.count('arr_del15')).alias('missing_delays'),
    (F.count('*') - F.count('carrier_delay')).alias('missing_carrier_info')
).show()


[Stage 333:>                                                        (0 + 2) / 2]

+------------+--------------+--------------------+
|missing_year|missing_delays|missing_carrier_info|
+------------+--------------+--------------------+
|           0|             0|                   0|
+------------+--------------+--------------------+



These cells perform a data quality check. It calculates the count of missing values across key columns (years, delays, and carrier info).

In [46]:
# Count the total number of records in the dataset.
df.select(F.count("*").alias("total_records")).show()


+-------------+
|total_records|
+-------------+
|       416462|
+-------------+



This confirms the total volume of the dataset, which contains 416,462 records.

In [47]:
# Calculate the average delay minutes per incident type.
# The filter removes rows where any incident count is zero to avoid division by zero.
df.filter(
    (F.col("carrier_ct") > 0) & (F.col("weather_ct") > 0) & 
    (F.col("nas_ct") > 0) & (F.col("late_aircraft_ct") > 0)
).select(
    # Total delay minutes divided by total incident count gives average minutes per incident.
    F.round(F.sum("carrier_delay") / F.sum("carrier_ct"), 2).alias("avg_carrier_delay_min"),
    F.round(F.sum("weather_delay") / F.sum("weather_ct"), 2).alias("avg_weather_delay_min"),
    F.round(F.sum("nas_delay") / F.sum("nas_ct"), 2).alias("avg_nas_delay_min"),
    F.round(F.sum("late_aircraft_delay") / F.sum("late_aircraft_ct"), 2).alias("avg_late_aircraft_delay_min")
).show()


[Stage 339:>                                                        (0 + 2) / 2]

+---------------------+---------------------+-----------------+---------------------------+
|avg_carrier_delay_min|avg_weather_delay_min|avg_nas_delay_min|avg_late_aircraft_delay_min|
+---------------------+---------------------+-----------------+---------------------------+
|                65.46|                90.55|            47.32|                      67.52|
+---------------------+---------------------+-----------------+---------------------------+



This analysis calculates the average duration (in minutes) for different delay types. Interestingly, while NAS (National Airspace System) delays are frequent, weather delays show the highest average duration at approximately 90.55 minutes.

In [48]:
# Find airports with the highest delay probability.
# Delay probability is delayed arrivals divided by total arrivals, converted to a percentage.
df.groupBy("airport", "airport_name") \
    .agg(
        F.sum("arr_flights").alias("total_flights"),
        ((F.sum("arr_del15") / F.sum("arr_flights")) * 100).alias("delay_probability")
    ) \
    # Keep only airports with enough flights to make the ranking meaningful.
    .filter(F.col("total_flights") > 1000) \
    .orderBy(F.desc("delay_probability")) \
    .limit(10) \
    .show(truncate=False)


[Stage 342:============================>                            (1 + 1) / 2]

+-------+--------------------------------------------------------+-------------+------------------+
|airport|airport_name                                            |total_flights|delay_probability |
+-------+--------------------------------------------------------+-------------+------------------+
|DUT    |Unalaska, AK: Unalaska Airport                          |2074.0       |37.126325940212155|
|ILG    |Wilmington, DE: New Castle                              |2121.0       |32.43752946723244 |
|MCN    |Macon, GA: Middle Georgia Regional                      |6483.0       |31.621163041801637|
|PQI    |Presque Isle/Houlton, ME: Presque Isle International    |4083.0       |28.704384031349502|
|BQN    |Aguadilla, PR: Rafael Hernandez                         |35615.0      |28.63400252702513 |
|SFB    |Sanford, FL: Orlando Sanford International              |76359.0      |28.0647991723307  |
|ISO    |Kinston, NC: Kinston Regional Jetport at Stallings Field|1456.0       |27.953296703296704|


This identifies the top 10 airports with the highest delay probability, filtered for those with at least 1,000 flights to ensure statistical relevance. Airports like Unalaska (DUT) and Wilmington (ILG) appear at the top of this list.

In [49]:
# Build a smaller modeling DataFrame with calculated features.
# Rows with zero flights or zero delay are removed to avoid invalid ratios.
model_ready_df = df.filter((F.col("arr_flights") > 0) & (F.col("arr_delay") > 0)) \
    .select(
        "year", "month", "carrier", "airport",
        # delay_ratio measures the share of flights delayed by at least 15 minutes.
        (F.col("arr_del15") / F.col("arr_flights")).alias("delay_ratio"),
        # cascade_factor measures the share of total delay minutes caused by late aircraft.
        (F.col("late_aircraft_delay") / F.col("arr_delay")).alias("cascade_factor")
    )

# Preview the engineered features.
model_ready_df.show(5)


+----+-----+-------+-------+-------------------+-------------------+
|year|month|carrier|airport|        delay_ratio|     cascade_factor|
+----+-----+-------+-------+-------------------+-------------------+
|2025|   11|     9E|    ABE| 0.1724137931034483| 0.6423576423576424|
|2025|   11|     9E|    ABY| 0.2727272727272727|0.25252525252525254|
|2025|   11|     9E|    AEX| 0.1724137931034483|0.36685082872928176|
|2025|   11|     9E|    AGS| 0.2413793103448276|0.40691489361702127|
|2025|   11|     9E|    ALB|0.35714285714285715| 0.5435308343409916|
+----+-----+-------+-------+-------------------+-------------------+
only showing top 5 rows



In this step, we create two new metrics for future modeling: the delay_ratio (percentage of flights delayed) and the cascade_factor (how much of the total delay is attributed to late-arriving aircraft).

In [50]:
# Measure how often each delay cause appears among delayed flights.
# Each cause count is divided by the total number of delayed arrivals.
df.filter(F.col("arr_del15") > 0) \
    .select(
        (F.sum("carrier_ct") / F.sum("arr_del15")).alias("carrier_freq_prop"),
        (F.sum("weather_ct") / F.sum("arr_del15")).alias("weather_freq_prop"),
        (F.sum("nas_ct") / F.sum("arr_del15")).alias("nas_freq_prop"),
        (F.sum("late_aircraft_ct") / F.sum("arr_del15")).alias("late_aircraft_freq_prop")
    ).show()


[Stage 346:>                                                        (0 + 2) / 2]

+------------------+-------------------+------------------+-----------------------+
| carrier_freq_prop|  weather_freq_prop|     nas_freq_prop|late_aircraft_freq_prop|
+------------------+-------------------+------------------+-----------------------+
|0.2955698196412167|0.03627120045895157|0.3180631176583486|     0.3476518392179479|
+------------------+-------------------+------------------+-----------------------+



In [51]:
# Analyze late-aircraft ripple effects by airport.
# total_ripple_minutes captures total late-aircraft delay minutes.
# ripple_intensity captures late-aircraft delay as a share of all arrival delay minutes.
df.filter(F.col("arr_delay") > 0) \
    .groupBy("airport", "airport_name") \
    .agg(
        F.sum("late_aircraft_delay").alias("total_ripple_minutes"),
        F.avg(F.col("late_aircraft_delay") / F.col("arr_delay")).alias("ripple_intensity")
    ) \
    .orderBy(F.desc("total_ripple_minutes")) \
    .limit(10) \
    .show(truncate=False)


[Stage 349:>                                                        (0 + 2) / 2]

+-------+------------------------------------------------------+--------------------+-------------------+
|airport|airport_name                                          |total_ripple_minutes|ripple_intensity   |
+-------+------------------------------------------------------+--------------------+-------------------+
|ORD    |Chicago, IL: Chicago O'Hare International             |3.9160337E7         |0.28132497962738723|
|DFW    |Dallas/Fort Worth, TX: Dallas/Fort Worth International|3.3256758E7         |0.30690667847821673|
|ATL    |Atlanta, GA: Hartsfield-Jackson Atlanta International |3.2756892E7         |0.3154877772527809 |
|DEN    |Denver, CO: Denver International                      |2.4167433E7         |0.3296206063719833 |
|LAX    |Los Angeles, CA: Los Angeles International            |1.9868628E7         |0.3292844329976183 |
|IAH    |Houston, TX: George Bush Intercontinental/Houston     |1.6279714E7         |0.2936082702186832 |
|CLT    |Charlotte, NC: Charlotte Douglas Inte

These cells analyze the Ripple Effect. Cell 11 specifically ranks major hubs like Chicago O'Hare (ORD) and Dallas/Fort Worth (DFW) by "total ripple minutes," highlighting where a single delay causes the most significant secondary delays.

In [52]:
# Aggregate delay metrics by year and month to inspect time trends.
temporal_df = df.groupBy("year", "month") \
    .agg(
        F.avg("arr_delay").alias("avg_delay_min"),
        F.sum("arr_del15").alias("total_delay_incidents"),
        F.sum("arr_flights").alias("total_flights")
    ) \
    .orderBy("year", "month")

# Preview the monthly trend DataFrame.
temporal_df.show(5)


[Stage 352:============================>                            (1 + 1) / 2]

+----+-----+------------------+---------------------+-------------+
|year|month|     avg_delay_min|total_delay_incidents|total_flights|
+----+-----+------------------+---------------------+-------------+
|2003|    6|  3395.25140562249|              89441.0|     536496.0|
|2003|    7| 4590.197758206566|             104579.0|     558568.0|
|2003|    8| 4682.253408179631|             106326.0|     556984.0|
|2003|    9|2575.9815261044178|              67386.0|     527714.0|
|2003|   10| 2455.594703049759|              69394.0|     552370.0|
+----+-----+------------------+---------------------+-------------+
only showing top 5 rows



In [53]:
# Create a window ordered by time within each airport-carrier combination.
# This lets us compare each record to previous months for the same route context.
window_spec = Window.partitionBy("airport", "carrier").orderBy("year", "month")

# Add lag-based features for previous-month and rolling three-month delay behavior.
df_with_lags = df.withColumn("prev_month_delay", F.lag("arr_del15", 1).over(window_spec)) \
                 .withColumn("rolling_3mo_delay", F.avg("arr_del15").over(window_spec.rowsBetween(-3, -1)))


In [54]:
# Compare weather delay patterns by month.
# Average weather delay shows typical severity, while standard deviation shows volatility.
df_with_lags.groupBy("month") \
    .agg(
        F.avg("weather_delay").alias("avg_weather_delay"),
        F.stddev("weather_delay").alias("weather_volatility")
    ) \
    .orderBy(F.desc("avg_weather_delay")) \
    .show()


[Stage 355:>                                                        (0 + 2) / 2]

+-----+------------------+------------------+
|month| avg_weather_delay|weather_volatility|
+-----+------------------+------------------+
|    7| 343.0077492974538|1170.3440603128663|
|    6| 327.1162445781447|1137.5667867180948|
|   12| 277.3639376276001|1057.4353898594852|
|    8| 277.0800270133378| 952.3061740106588|
|    1| 277.0265543103698|1074.2290378109185|
|    5|241.99869645079102|1030.3859394081833|
|    2| 233.5664464993395| 909.2596143354214|
|    3|182.97384915891664| 751.5812985077213|
|    4| 178.6761686229518| 738.4743983128502|
|    9|141.62326290814596| 554.9287785836673|
|   11|135.10727510966785| 601.9083919906113|
|   10| 128.3183934641643| 625.8553272315236|
+-----+------------------+------------------+



This cell ranks the months based on their average weather delay across every year. We can see that the months of June, July, and December have had the highest average weather delay. It also shows their weather volatility.

In [55]:
# Compare monthly delay and cancellation ratios.
# Both ratios are normalized by total arriving flights for that month.
df.groupBy("month") \
    .agg(
        (F.sum("arr_del15") / F.sum("arr_flights")).alias("delay_ratio"),
        (F.sum("arr_cancelled") / F.sum("arr_flights")).alias("cancellation_ratio")
    ) \
    .orderBy(F.desc("delay_ratio")) \
    .show()


[Stage 358:============================>                            (1 + 1) / 2]

+-----+-------------------+--------------------+
|month|        delay_ratio|  cancellation_ratio|
+-----+-------------------+--------------------+
|    6|0.23312286177549496|0.016995686585123643|
|    7|0.23031488132956757| 0.01776844011932469|
|   12| 0.2265809965746147|  0.0212411638644161|
|    8|0.20448706908286843|0.016765566165599547|
|    1| 0.1963668412475567|0.028912991018658598|
|    2| 0.1953154182070338|0.027412513460730842|
|    5| 0.1908318852249711| 0.01308758916817654|
|    3|0.18770203429318644|0.025066349248680543|
|    4|0.17806880725572413|0.024344872558568632|
|   10|0.16334121409696684| 0.01038959600774967|
|   11|0.16012084470334745|0.009483156251411173|
|    9|0.14963573021020232|0.013139415558873196|
+-----+-------------------+--------------------+



In [56]:
# Rank years by overall delay ratio.
# This identifies which years had the highest share of delayed flights.
df.groupBy("year") \
    .agg(
        F.sum("arr_del15").alias("total_delayed_flights"),
        F.sum("arr_flights").alias("total_flights"),
        (F.sum("arr_del15") / F.sum("arr_flights")).alias("delay_ratio")
    ) \
    .orderBy(F.desc("delay_ratio")) \
    .limit(10) \
    .show()


[Stage 361:============================>                            (1 + 1) / 2]

+----+---------------------+-------------+-------------------+
|year|total_delayed_flights|total_flights|        delay_ratio|
+----+---------------------+-------------+-------------------+
|2007|            1804028.0|    7455458.0|0.24197413492236158|
|2006|            1615537.0|    7141922.0|0.22620479473172628|
|2008|            1524735.0|    7009726.0| 0.2175170612945499|
|2025|            1515201.0|    7091783.0|0.21365586059246314|
|2014|            1240528.0|    5819811.0|0.21315606297180442|
|2005|            1466065.0|    7140595.0|0.20531412298274865|
|2022|            1426305.0|    7013508.0|0.20336541998668856|
|2024|            1531080.0|    7546968.0|0.20287352483805418|
|2023|            1464539.0|    7278739.0|0.20120779162434593|
|2004|            1421391.0|    7129270.0|0.19937398920226054|
+----+---------------------+-------------+-------------------+



This shows the years ranked based on their delay ratio. So it shows the top 10 delay ratios. We can see 2006, 2007, and 2008 had the highest delay ratios.

In [57]:
# Compare airline carriers by delay ratio and internally caused delay share.
# The flight-count filter removes very small carriers from the ranking.
df.groupBy("carrier_name") \
    .agg(
        (F.sum("arr_del15") / F.sum("arr_flights")).alias("delay_ratio"),
        (F.sum("carrier_delay") / F.sum("arr_delay")).alias("internal_delay_ratio"),
        F.sum("arr_flights").alias("total_flights_count")
    ) \
    .filter(F.col("total_flights_count") > 1000) \
    .orderBy(F.desc("delay_ratio")) \
    .limit(10) \
    # Drop helper count column so the output focuses on the ratios.
    .drop("total_flights_count") \
    .show()


[Stage 364:============================>                            (1 + 1) / 2]

+--------------------+-------------------+--------------------+
|        carrier_name|        delay_ratio|internal_delay_ratio|
+--------------------+-------------------+--------------------+
|Peninsula Airways...|0.31453362255965295| 0.39321353833382333|
|   Frontier Airlines|0.26359599662269934| 0.29191984260288906|
|Trans States Airl...|0.24601623825085064|  0.2941755296932184|
|Atlantic Southeas...|0.24466134918919946|  0.3803589851523679|
|     JetBlue Airways| 0.2440392301557146|  0.3227148910145084|
|Commutair Aka Cha...|0.24314902475191058|  0.2827876212743289|
|       Allegiant Air|0.23815437815358712| 0.32699280855899476|
|    Spirit Air Lines|0.23633109923471615| 0.19554874571374736|
|         Comair Inc.|0.22320746981019668| 0.39753005690925675|
|Northwest Airline...|0.22042211357414088|  0.3972576936580297|
+--------------------+-------------------+--------------------+



This shows which carrier had the highest delay ratios. It shows the top 10 carriers. It is ranked in decreasing order. 

In [58]:
# Calculate average delay minutes per delayed flight.
# arr_del15 is the count of flights delayed by at least 15 minutes.
df_with_avg = df.withColumn("avg_delay", F.col("arr_delay") / F.col("arr_del15"))

# Assign severity buckets based on average delay duration.
df_buckets = df_with_avg.withColumn("severity_bucket", 
    F.when(F.col("avg_delay") <= 15, "0–15 min")
     .when((F.col("avg_delay") > 15) & (F.col("avg_delay") <= 60), "15–60 min")
     .when((F.col("avg_delay") > 60) & (F.col("avg_delay") <= 180), "1–3 hrs")
     .otherwise("3+ hrs")
)

# Calculate total delayed flights so each bucket can be converted into a percentage.
total_delayed_flights = df.select(F.sum("arr_del15")).collect()[0][0]

# Aggregate by severity bucket and calculate the relative frequency percentage.
bucket_analysis = df_buckets.groupBy("severity_bucket") \
    .agg(F.sum("arr_del15").alias("flight_count")) \
    .withColumn("relative_frequency_pct", 
                F.round((F.col("flight_count") / total_delayed_flights) * 100, 2)) \
    .orderBy(F.col("relative_frequency_pct").desc())

# Show which delay-severity buckets are most common.
bucket_analysis.show()


[Stage 370:============================>                            (1 + 1) / 2]

+---------------+------------+----------------------+
|severity_bucket|flight_count|relative_frequency_pct|
+---------------+------------+----------------------+
|      15–60 min| 1.6315152E7|                 56.15|
|        1–3 hrs| 1.2721108E7|                 43.78|
|         3+ hrs|     17161.0|                  0.06|
|       0–15 min|       667.0|                   0.0|
+---------------+------------+----------------------+



This shows the frequency percentages of the delay times grouped into severity buckets. The severity bucket with the highest frequency percentage is between 15-60 minutes.

In [59]:
# Calculate total delay minutes across the full dataset.
# This denominator is used to find each cause's share of total delay time.
total_delay_minutes = df.select(F.sum("arr_delay")).collect()[0][0]

# Calculate the percentage share of total delay minutes for each delay cause.
cause_ownership = df.select(
    (F.sum("carrier_delay") / total_delay_minutes * 100).alias("carrier_delay_pct"),
    (F.sum("weather_delay") / total_delay_minutes * 100).alias("weather_delay_pct"),
    (F.sum("nas_delay") / total_delay_minutes * 100).alias("nas_delay_pct"),
    (F.sum("security_delay") / total_delay_minutes * 100).alias("security_delay_pct"),
    (F.sum("late_aircraft_delay") / total_delay_minutes * 100).alias("late_aircraft_delay_pct")
)

# Display cause-level ownership of delay minutes.
cause_ownership.show()


[Stage 376:============================>                            (1 + 1) / 2]

+-----------------+-----------------+------------------+-------------------+-----------------------+
|carrier_delay_pct|weather_delay_pct|     nas_delay_pct| security_delay_pct|late_aircraft_delay_pct|
+-----------------+-----------------+------------------+-------------------+-----------------------+
|31.42396854573916|5.411808509756749|24.469545723663753|0.16856510779125292|     38.525744982645335|
+-----------------+-----------------+------------------+-------------------+-----------------------+



In [60]:
# Calculate total delayed-flight incidents across the dataset.
total_incidents = df.select(F.sum("arr_del15")).collect()[0][0]

# Calculate how frequently each delay cause appears as a percentage of all delay incidents.
frequency_impact = df.select(
    (F.sum("carrier_ct") / total_incidents * 100).alias("Carrier_Freq_Pct"),
    (F.sum("weather_ct") / total_incidents * 100).alias("Weather_Freq_Pct"),
    (F.sum("nas_ct") / total_incidents * 100).alias("NAS_Freq_Pct"),
    (F.sum("security_ct") / total_incidents * 100).alias("Security_Freq_Pct"),
    (F.sum("late_aircraft_ct") / total_incidents * 100).alias("Late_Aircraft_Freq_Pct")
)

# Display cause-level frequency percentages.
frequency_impact.show()


[Stage 382:============================>                            (1 + 1) / 2]

+------------------+------------------+-----------------+-------------------+----------------------+
|  Carrier_Freq_Pct|  Weather_Freq_Pct|     NAS_Freq_Pct|  Security_Freq_Pct|Late_Aircraft_Freq_Pct|
+------------------+------------------+-----------------+-------------------+----------------------+
|29.556981964121672|3.6271200458951567|31.80631176583486|0.24440309398112975|     34.76518392179479|
+------------------+------------------+-----------------+-------------------+----------------------+



In [61]:
# Calculate average delay duration when a specific cause occurs.
# Delay minutes are divided by the number of incidents for each cause.
severity_impact = df.select(
    (F.sum("carrier_delay") / F.sum("carrier_ct")).alias("Avg_Carrier_Delay_Min"),
    (F.sum("weather_delay") / F.sum("weather_ct")).alias("Avg_Weather_Delay_Min"),
    (F.sum("nas_delay") / F.sum("nas_ct")).alias("Avg_NAS_Delay_Min"),
    # This uses nas_ct in the denominator; change to security_ct if you want security-specific incident severity.
    (F.sum("security_delay") / F.sum("nas_ct")).alias("Avg_Security_Delay_Min"),
    (F.sum("late_aircraft_delay") / F.sum("late_aircraft_ct")).alias("Avg_Late_Aircraft_Delay_Min")
)

# Display average delay minutes by cause.
severity_impact.show()


[Stage 385:============================>                            (1 + 1) / 2]

+---------------------+---------------------+-----------------+----------------------+---------------------------+
|Avg_Carrier_Delay_Min|Avg_Weather_Delay_Min|Avg_NAS_Delay_Min|Avg_Security_Delay_Min|Avg_Late_Aircraft_Delay_Min|
+---------------------+---------------------+-----------------+----------------------+---------------------------+
|    64.42800638795036|    90.41786331746142|46.62151084587357|    0.3211649325196004|          67.15529519565357|
+---------------------+---------------------+-----------------+----------------------+---------------------------+



In [62]:
# Import tools for feature vector creation, linear regression, and model evaluation.
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Select count-based delay cause columns as model predictors.
feature_cols = ["carrier_ct", "weather_ct", "nas_ct", "security_ct", "late_aircraft_ct"]

# Create a clean modeling DataFrame with predictors and the target variable.
# dropna() removes rows with missing values before fitting the model.
model_df = df.select(feature_cols + ["arr_delay"]).dropna()

# Combine the predictor columns into one vector column named "features".
# PySpark ML models require all features to be stored in a single vector column.
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
vector_df = assembler.transform(model_df)

# Split the data into training and testing sets.
# The seed makes the split reproducible.
train_data, test_data = vector_df.randomSplit([0.8, 0.2], seed=42)

# Define and train a linear regression model to predict total arrival delay minutes.
lr = LinearRegression(featuresCol="features", labelCol="arr_delay")
lr_model = lr.fit(train_data)

# Use the trained model to generate predictions on the test data.
predictions = lr_model.transform(test_data)

# Evaluate the regression model using R-squared.
# R-squared measures how much variance in arr_delay is explained by the model.
evaluator = RegressionEvaluator(labelCol="arr_delay", predictionCol="prediction", metricName="r2")
r2 = evaluator.evaluate(predictions)

# Print the model's test-set R-squared score.
print(f"R-squared (R2) on test data: {r2:.4f}")

# Print each feature coefficient to interpret the linear model.
# Larger positive coefficients indicate a stronger association with total delay minutes.
print("Weight of each cause (Coefficients):")
for col, coef in zip(feature_cols, lr_model.coefficients):
    print(f"  {col}: {coef:.2f}")


26/05/03 18:33:23 WARN Instrumentation: [8d174e0b] regParam is zero, which might cause numerical instability and overfitting.
[Stage 390:============================>                            (1 + 1) / 2]

R-squared (R2) on test data: 0.9378
Weight of each cause (Coefficients):
  carrier_ct: 62.39
  weather_ct: 153.00
  nas_ct: 59.96
  security_ct: -766.39
  late_aircraft_ct: 67.18


This PySpark script implements a supervised machine learning workflow to predict total airline delay minutes based on specific delay categories. It utilizes a VectorAssembler to consolidate features like weather and carrier counts into a single input format required for training. The data is split into 80/20 training and testing sets to evaluate performance, achieving a high R-squared value of 0.9378. Finally, the code outputs the predictive weights of each cause, revealing how much each specific factor contributes to the overall flight delay.

In [63]:
# Import pipeline and feature engineering tools for mixed categorical/numerical features.
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Define categorical and numerical features for the regression model.
# carrier, airport, and month represent airline, location, and seasonality effects.
categorical_cols = ["carrier", "airport", "month"]

# arr_flights represents scheduled volume; weather_ct represents weather-related incidents.
numerical_cols = ["arr_flights", "weather_ct"]

# Convert categorical values into numeric category indexes.
# handleInvalid="keep" prevents the pipeline from failing on unseen/invalid categories.
indexers = [StringIndexer(inputCol=col, outputCol=col+"_index", handleInvalid="keep") 
            for col in categorical_cols]

# Convert category indexes into one-hot encoded vectors.
# This avoids treating categorical IDs as ordered numeric values.
encoders = [OneHotEncoder(inputCol=col+"_index", outputCol=col+"_vec") 
            for col in categorical_cols]

# Assemble encoded categorical features and numerical features into one feature vector.
assembler_inputs = [col+"_vec" for col in categorical_cols] + numerical_cols
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")

# Define the linear regression model with arr_delay as the prediction target.
lr = LinearRegression(labelCol="arr_delay", featuresCol="features")

# Build the end-to-end machine learning pipeline.
# The pipeline runs indexing, encoding, vector assembly, and model training in sequence.
pipeline = Pipeline(stages=indexers + encoders + [assembler, lr])

# Split the raw DataFrame into training and testing sets.
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

# Fit the full pipeline on the training data.
model = pipeline.fit(train_data)

# Generate predictions on the test data.
predictions = model.transform(test_data)

# Evaluate model error using RMSE.
# RMSE is in the same unit as the target variable: delay minutes.
evaluator = RegressionEvaluator(labelCol="arr_delay", metricName="rmse")
rmse = evaluator.evaluate(predictions)

# Print the RMSE and preview a few predictions against actual delay values.
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
predictions.select("carrier", "airport", "arr_delay", "prediction").show(5)


26/05/03 18:33:51 WARN Instrumentation: [e81acc83] regParam is zero, which might cause numerical instability and overfitting.
26/05/03 18:33:57 WARN Instrumentation: [e81acc83] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
                                                                                

Root Mean Squared Error (RMSE): 5947.97


[Stage 403:>                                                        (0 + 1) / 1]

+-------+-------+---------+------------------+
|carrier|airport|arr_delay|        prediction|
+-------+-------+---------+------------------+
|     AA|    AUS|   8245.0| 8893.429715015314|
|     AA|    BOS|  10338.0| 13956.24620333321|
|     AA|    CLE|    672.0|  1175.96014065913|
|     AA|    DCA|   6103.0|11847.286976385567|
|     AA|    ELP|   1139.0|1809.2802981225793|
+-------+-------+---------+------------------+
only showing top 5 rows



This code utilizes a PySpark Pipeline to automate the complex process of transforming categorical data, like carrier and airport codes, into a machine-learning-ready format. It employs StringIndexer and OneHotEncoder to convert text labels into binary vectors, which are then combined with numerical proxies for weather and flight volume. By bundling these steps with a LinearRegression model, the pipeline ensures consistent data handling and prevents leakage during the training and testing phases. The script ultimately measures the model's accuracy using Root Mean Squared Error (RMSE) to determine the average deviation between the predicted and actual delay times.

In [64]:
# Calculate the average predicted delay per flight.
# Dividing the model prediction by arriving flights normalizes delay by airport/carrier volume.
predictions.withColumn(
    "avg_delay_per_flight", 
    F.col("prediction") / F.col("arr_flights")
).select("carrier", "airport", "avg_delay_per_flight").show(5)


[Stage 404:>                                                        (0 + 1) / 1]

+-------+-------+--------------------+
|carrier|airport|avg_delay_per_flight|
+-------+-------+--------------------+
|     AA|    AUS|  13.874305327636996|
|     AA|    BOS|  15.104162557719924|
|     AA|    CLE|  13.834825184225059|
|     AA|    DCA|  14.154464726864477|
|     AA|    ELP|   22.33679380398246|
+-------+-------+--------------------+
only showing top 5 rows



This code calculates a normalized delay metric by dividing the model's total predicted delay by the number of arriving flights for each specific data point. It then displays the first five records, highlighting the carrier and airport to help identify which routes have the highest predicted delay per flight.

In [65]:
# Import logistic regression and binary classification evaluation tools.
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import VectorAssembler, StringIndexer, VectorIndexer

# Create a delay ratio for each record.
# This measures the percentage of arriving flights delayed by at least 15 minutes.
df_class = df.withColumn("delay_ratio", F.col("arr_del15") / F.col("arr_flights"))

# Create a binary label for high-risk delay situations.
# A record is high risk if more than 20% of flights were delayed.
df_class = df_class.withColumn("is_high_risk", F.when(F.col("delay_ratio") > 0.2, 1).otherwise(0))

# Define the features used for classification.
feature_cols = ["carrier", "airport", "month"]

# Convert string columns into numeric indexes so they can be used in the model.
indexers = [StringIndexer(inputCol=col, outputCol=col+"_idx").fit(df_class) for col in ["carrier", "airport"]]

# Combine indexed carrier, indexed airport, and month into a single features vector.
assembler = VectorAssembler(inputCols=["carrier_idx", "airport_idx", "month"], outputCol="features")

# Split the classification data into training and testing sets.
train_data, test_data = df_class.randomSplit([0.8, 0.2], seed=42)

# Define the logistic regression classifier.
lr = LogisticRegression(labelCol="is_high_risk", featuresCol="features")

# Import Pipeline and chain feature preparation with model training.
from pyspark.ml import Pipeline
pipeline = Pipeline(stages=indexers + [assembler, lr])

# Train the classification pipeline and generate test predictions.
model = pipeline.fit(train_data)
predictions = model.transform(test_data)

# Evaluate the classifier using area under the ROC curve.
# AUC measures how well the model separates high-risk and non-high-risk records.
evaluator = BinaryClassificationEvaluator(labelCol="is_high_risk", metricName="areaUnderROC")
auc = evaluator.evaluate(predictions)

# Print the AUC and preview predicted probabilities/classes.
print(f"Area Under ROC (Accuracy Metric): {auc:.4f}")
predictions.select("carrier", "airport", "is_high_risk", "probability", "prediction").show(5)


Area Under ROC (Accuracy Metric): 0.5568


[Stage 430:>                                                        (0 + 1) / 1]

+-------+-------+------------+--------------------+----------+
|carrier|airport|is_high_risk|         probability|prediction|
+-------+-------+------------+--------------------+----------+
|     AA|    AUS|           1|[0.54749489138216...|       0.0|
|     AA|    BOS|           1|[0.54919952170596...|       0.0|
|     AA|    CLE|           0|[0.54863143789530...|       0.0|
|     AA|    DCA|           0|[0.54578914620512...|       0.0|
|     AA|    ELP|           1|[0.58467079602730...|       0.0|
+-------+-------+------------+--------------------+----------+
only showing top 5 rows



In [66]:
# Import gradient-boosted tree regression and evaluation tools.
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import RegressionEvaluator

# Use delay-cause count columns as predictors for total arrival delay minutes.
feature_cols = ["carrier_ct", "weather_ct", "nas_ct", "security_ct", "late_aircraft_ct"]

# Combine predictor columns into the single vector column required by PySpark ML.
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# Build the final modeling DataFrame and remove rows with missing predictor/target values.
final_df = assembler.transform(df.select(feature_cols + ["arr_delay"]).dropna())

# Split the data into training and testing sets.
train_data, test_data = final_df.randomSplit([0.8, 0.2], seed=42)

# Initialize and train a Gradient-Boosted Trees regression model.
# maxIter controls the number of boosting iterations/trees.
gbt = GBTRegressor(featuresCol="features", labelCol="arr_delay", maxIter=20)
gbt_model = gbt.fit(train_data)

# Generate predictions and evaluate the model with R-squared.
gbt_predictions = gbt_model.transform(test_data)
evaluator = RegressionEvaluator(labelCol="arr_delay", metricName="r2")

# Print the Gradient-Boosted Model's R-squared score.
print(f"GBM R-squared: {evaluator.evaluate(gbt_predictions):.4f}")


[Stage 635:============================>                            (1 + 1) / 2]

GBM R-squared: 0.7407


Summary of Machine Learning ResultsPredictive Accuracy: The Linear Regression model emerged as the most effective for predicting total delay minutes, achieving a remarkably high R-squared (R2) of 0.9378 on the test data.  Key Drivers: The model coefficients highlight that Weather has the highest individual impact weight (153.00), meaning a single weather-related incident adds significantly more delay time than any other cause. Late Aircraft (67.18) and Carrier issues (62.39) follow as the next most influential predictors.  Risk Classification: The Logistic Regression model attempted to classify "High Risk" flights (delays > 20% ratio), resulting in an Area Under ROC (AUC) of 0.5568. This suggests that while raw delay minutes are predictable, binary risk status is more volatile and influenced by factors outside the basic carrier/airport/month features.  Model Complexity: Implementing a Gradient Boosted Tree (GBT) Regressor yielded an R-squared of 0.7407, showing that while non-linear models are powerful, the relationship between delay causes and total minutes in this dataset is highly linear and better captured by standard regression. 

Conclusion
The modeling phase of the project confirms that airline delays are not random occurrences but structured events that can be predicted with high precision using the right features. By leveraging the Spark ML Pipeline, we established that:  

Weather is the ultimate "force multiplier" for delay duration, even if it is not the most frequent cause.  

Linear relationships dominate: The extreme accuracy of the Linear Regression model indicates that total delay is essentially a cumulative sum of its parts, allowing stakeholders to estimate total downtime simply by monitoring incident counts.  

Future Directions: The failed import of the MultilayerPerceptronRegressor and the lower AUC in classification suggest that future iterations could benefit from more granular features—such as real-time weather data or aircraft tail-number history—to better predict high-risk operational windows.  

Overall, these models provide a robust framework for airlines to forecast labor and resource needs based on predicted delay intensities across their network.